<a href="https://colab.research.google.com/github/imagra93/ML-course-labs/blob/main/machina_learning/lab%20-%20XGBoost%20and%20Gradient%20Boosting.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# XGBoost and Gradient Boosting

**Theory:** [notes_ML/07_bagging_random_forest_boosting.ipynb](../notes_ML/07_bagging_random_forest_boosting.ipynb) (bagging vs. boosting, AdaBoost, gradient boosting as gradient descent in function space). This lab builds the intuition with plots and then uses XGBoost in practice.

**Boosting** adds many small trees one after another. Each new tree is fitted to what the current model still gets wrong, and is added with a small weight $\eta$ (the learning rate):

$$F_T(\mathbf{x}) = F_0 + \eta \sum_{t=1}^{T} f_t(\mathbf{x})$$

**XGBoost** (Chen & Guestrin, 2016) is a fast, regularised implementation of gradient boosting. Compared with plain gradient boosting it penalises big trees and large leaf values, uses the curvature of the loss to choose leaf values, and handles missing values and large datasets efficiently.

Contents:
1. Gradient boosting from scratch on a toy problem
2. Regression (Diabetes)
3. Binary classification (Breast Cancer)
4. Multiclass classification (Wine)
5. Cross-validation and hyperparameter search
6. Learning rate, number of trees and early stopping
7. Feature importance

Useful links: [XGBoost documentation](https://xgboost.readthedocs.io/en/latest/index.html), [parameters](https://xgboost.readthedocs.io/en/latest/parameter.html), [Python package](https://xgboost.readthedocs.io/en/latest/python/python_intro.html).

In [ ]:
!pip3 install --upgrade xgboost

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import uniform, randint

from sklearn.datasets import load_breast_cancer, load_diabetes, load_wine
from sklearn.metrics import (accuracy_score, confusion_matrix, mean_squared_error,
                             r2_score, roc_auc_score, roc_curve)
from sklearn.model_selection import (cross_val_score, KFold, RandomizedSearchCV,
                                     train_test_split)
from sklearn.tree import DecisionTreeRegressor

import xgboost as xgb

plt.style.use('seaborn-v0_8-whitegrid')
print(f"xgboost version: {xgb.__version__}")

## 1. Gradient boosting from scratch

A 1D toy problem makes boosting easy to see. With squared loss, the "error to correct" is just the residual $y - F(x)$ (see notes 07, section 6).

In code:
1. Start with a constant prediction $F_0 = \text{mean}(y)$.
2. Compute the residuals $r = y - F$.
3. Fit a small tree (depth 2) to the residuals.
4. Update $F \leftarrow F + \eta \cdot \text{tree}(x)$ and repeat from step 2.

In [ ]:
rng = np.random.default_rng(0)
x = np.sort(rng.uniform(0, 6, 200))
y = np.sin(x) + 0.3 * x + rng.normal(0, 0.3, 200)
X_toy = x[:, None]                        # sklearn/xgboost expect a 2D array

eta, n_trees = 0.1, 100
F = np.full_like(y, y.mean())             # step 1: constant start
snapshots, train_mse = {}, []

for t in range(1, n_trees + 1):
    residuals = y - F                                              # step 2
    tree = DecisionTreeRegressor(max_depth=2).fit(X_toy, residuals)  # step 3
    F = F + eta * tree.predict(X_toy)                              # step 4
    train_mse.append(np.mean((y - F) ** 2))
    if t in (1, 5, 20, 100):
        snapshots[t] = F.copy()

fig, axes = plt.subplots(1, 3, figsize=(16, 4))

# Left: the model after more and more trees
axes[0].scatter(x, y, s=10, color='k', alpha=0.4)
for t, F_t in snapshots.items():
    axes[0].plot(x, F_t, lw=2, label=f'after {t} trees')
axes[0].set_title(rf'Model after $t$ trees ($\eta$={eta}, depth 2)')
axes[0].legend()

# Middle: residuals at the start vs. at the end
axes[1].scatter(x, y - y.mean(), s=10, alpha=0.5, label='residuals at start')
axes[1].scatter(x, y - F, s=10, alpha=0.5, label=f'residuals after {n_trees} trees')
axes[1].axhline(0, color='k', lw=1)
axes[1].set_title('Residuals shrink towards noise')
axes[1].legend()

# Right: training error per round
axes[2].plot(range(1, n_trees + 1), train_mse, lw=2)
axes[2].axhline(0.3 ** 2, color='gray', ls='--', label='noise variance (0.09)')
axes[2].set_xlabel('number of trees'); axes[2].set_ylabel('training MSE')
axes[2].set_title('Training error per boosting round')
axes[2].legend()

plt.tight_layout()
plt.show()

Each tree only fixes a small part of the error, so the curve gets closer to the data step by step. After about 30 trees the training MSE drops below the noise variance (dashed line): the model has started fitting the noise, so the number of trees must be chosen with a validation set (section 6).

XGBoost does the same loop. For squared loss, the value it puts in a leaf is roughly

$$w = \frac{\text{sum of residuals in the leaf}}{\text{number of samples in the leaf} + \lambda}$$

that is, the mean residual shrunk towards 0 by the L2 penalty $\lambda$ (`reg_lambda`, default 1). Splits that improve the loss by less than $\gamma$ (`gamma`) are not made. With the same settings it gives almost the same fit as our loop:

In [ ]:
xgb_toy = xgb.XGBRegressor(n_estimators=n_trees, learning_rate=eta, max_depth=2)
xgb_toy.fit(X_toy, y)

print(f"From scratch training MSE: {train_mse[-1]:.4f}")
print(f"XGBoost      training MSE: {mean_squared_error(y, xgb_toy.predict(X_toy)):.4f}")

## 2. Regression: Diabetes dataset

The [Diabetes dataset](https://scikit-learn.org/stable/modules/generated/sklearn.datasets.load_diabetes.html) has 442 patients and 10 features (age, sex, BMI, blood pressure, 6 blood serum measurements). The target is a measure of disease progression one year later. We report RMSE (same units as the target) and $R^2$ (fraction of variance explained).

In [ ]:
diabetes = load_diabetes()
X, y = diabetes.data, diabetes.target
print(f"Samples: {X.shape[0]}, features: {X.shape[1]}")
print(f"Target range: [{y.min():.0f}, {y.max():.0f}]  (mean={y.mean():.1f})")
print(f"Feature names: {diabetes.feature_names}")

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

xgb_reg = xgb.XGBRegressor(objective="reg:squarederror", n_estimators=200,
                           learning_rate=0.1, max_depth=4, random_state=42)
xgb_reg.fit(X_train, y_train)

y_pred_train = xgb_reg.predict(X_train)
y_pred_test = xgb_reg.predict(X_test)

print(f"\nTrain RMSE: {np.sqrt(mean_squared_error(y_train, y_pred_train)):.1f}   R2: {r2_score(y_train, y_pred_train):.3f}")
print(f"Test  RMSE: {np.sqrt(mean_squared_error(y_test, y_pred_test)):.1f}   R2: {r2_score(y_test, y_pred_test):.3f}")

The training error is much lower than the test error: with 200 trees of depth 4 the model has memorised much of the training set. Sections 5 and 6 show how to choose the settings with cross-validation and early stopping.

## 3. Binary classification: Breast Cancer dataset

The [Breast Cancer Wisconsin dataset](https://scikit-learn.org/stable/modules/generated/sklearn.datasets.load_breast_cancer.html) has 569 samples and 30 features (measurements of cell nuclei). Target: malignant (0) or benign (1).

With `objective="binary:logistic"` the trees add up to a log-odds score $F(\mathbf{x})$, the predicted probability is $\sigma(F(\mathbf{x}))$, and the loss is the log-loss. Trees split on thresholds, so no feature scaling is needed.

In [ ]:
cancer = load_breast_cancer()
X, y = cancer.data, cancer.target
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)

xgb_clf = xgb.XGBClassifier(objective="binary:logistic", random_state=42)
xgb_clf.fit(X_train, y_train)

y_pred = xgb_clf.predict(X_test)                 # class labels
y_proba = xgb_clf.predict_proba(X_test)[:, 1]    # probability of class 1 (benign)

print(f"Train accuracy: {accuracy_score(y_train, xgb_clf.predict(X_train)):.3f}")
print(f"Test  accuracy: {accuracy_score(y_test, y_pred):.3f}")
print(f"Test  ROC AUC : {roc_auc_score(y_test, y_proba):.3f}")

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4))

# Confusion matrix on the test set
sns.heatmap(confusion_matrix(y_test, y_pred), annot=True, fmt='d', cmap='Blues', ax=ax1,
            xticklabels=cancer.target_names, yticklabels=cancer.target_names)
ax1.set_xlabel('Predicted'); ax1.set_ylabel('Actual')
ax1.set_title('Test confusion matrix')

# ROC curve on the test set
fpr, tpr, _ = roc_curve(y_test, y_proba)
ax2.plot(fpr, tpr, lw=2, label=f'XGBoost (AUC={roc_auc_score(y_test, y_proba):.3f})')
ax2.plot([0, 1], [0, 1], 'k--', lw=1, label='random guess')
ax2.set_xlabel('False positive rate'); ax2.set_ylabel('True positive rate')
ax2.set_title('Test ROC curve'); ax2.legend()

plt.tight_layout()
plt.show()

## 4. Multiclass classification: Wine dataset

The [Wine dataset](https://scikit-learn.org/stable/modules/generated/sklearn.datasets.load_wine.html) has 178 samples, 13 chemical features and 3 classes (grape cultivars).

With `objective="multi:softprob"` XGBoost grows one tree per class in each round, giving one score $F_k(\mathbf{x})$ per class, and turns the scores into probabilities with the softmax:
$$P(y=k \mid \mathbf{x}) = \frac{e^{F_k(\mathbf{x})}}{\sum_{j} e^{F_j(\mathbf{x})}}$$

In [ ]:
wine = load_wine()
X, y = wine.data, wine.target
print(f"Samples: {X.shape[0]}, features: {X.shape[1]}, classes: {np.unique(y)}")

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)

xgb_multi = xgb.XGBClassifier(objective="multi:softprob", random_state=42)
xgb_multi.fit(X_train, y_train)
y_pred = xgb_multi.predict(X_test)
acc = accuracy_score(y_test, y_pred)

fig, ax = plt.subplots(figsize=(5, 4))
sns.heatmap(confusion_matrix(y_test, y_pred), annot=True, fmt='d', cmap='Blues', ax=ax,
            xticklabels=wine.target_names, yticklabels=wine.target_names)
ax.set_xlabel('Predicted'); ax.set_ylabel('Actual')
ax.set_title(f'Wine: test confusion matrix (acc={acc:.3f})')
plt.tight_layout()
plt.show()

## 5. Cross-validation and hyperparameter search

One train/test split gives a noisy estimate. $k$-fold cross-validation trains and evaluates on $k$ different splits and averages the scores.

In [ ]:
def display_scores(scores):
    """Print CV scores: individual folds, mean and std."""
    print(f"RMSE per fold: {np.round(scores, 1)}")
    print(f"Mean RMSE    : {np.mean(scores):.1f}  (std {np.std(scores):.1f})")

X, y = diabetes.data, diabetes.target
kfold = KFold(n_splits=10, shuffle=True, random_state=42)

xgb_reg = xgb.XGBRegressor(objective="reg:squarederror", random_state=42)
scores = -cross_val_score(xgb_reg, X, y, cv=kfold, scoring="neg_root_mean_squared_error")
display_scores(scores)

The main XGBoost hyperparameters:

| Parameter | What it controls |
|---|---|
| `n_estimators`, `learning_rate` | number of trees and the weight of each tree (tuned together, section 6) |
| `max_depth` | size of each tree |
| `subsample`, `colsample_bytree` | fraction of rows / features used per tree (adds randomness, like a random forest) |
| `reg_lambda`, `gamma` | penalty on leaf values / minimum loss reduction to make a split |

**Random search** samples settings from a distribution for each parameter. It usually finds good values with far fewer fits than a full grid. The score below is the default regression score, $R^2$ (higher is better).

In [ ]:
def report_best_scores(results, n_top=3):
    """Print the top-n settings from a GridSearchCV / RandomizedSearchCV result."""
    for rank in range(1, n_top + 1):
        for i in np.flatnonzero(results['rank_test_score'] == rank):
            print(f"Rank {rank}: mean CV score {results['mean_test_score'][i]:.3f}"
                  f" (std {results['std_test_score'][i]:.3f})")
            params = {k: round(float(v), 3) if isinstance(v, float) else v for k, v in results['params'][i].items()}
            print(f"        params: {params}")

params = {
    "colsample_bytree": uniform(0.7, 0.3),   # 0.7 to 1.0
    "gamma": uniform(0, 0.5),
    "learning_rate": uniform(0.03, 0.3),     # 0.03 to 0.33
    "max_depth": randint(2, 6),              # 2 to 5
    "n_estimators": randint(100, 150),
    "subsample": uniform(0.6, 0.4),          # 0.6 to 1.0
}

search = RandomizedSearchCV(xgb.XGBRegressor(objective="reg:squarederror", random_state=42),
                            param_distributions=params, n_iter=50, cv=3,
                            random_state=42, n_jobs=1)
search.fit(X, y)
report_best_scores(search.cv_results_, n_top=3)

## 6. Learning rate, number of trees and early stopping

A smaller learning rate makes each tree count less, so more trees are needed, but the result often generalises a bit better. Too many trees overfit. We use the Diabetes data again, where overfitting is easy to see. **Early stopping** tracks a validation metric and stops when it has not improved for `early_stopping_rounds` rounds.

In code:
1. Split off a validation set.
2. Set a large `n_estimators` and pass `early_stopping_rounds` to the constructor.
3. Pass the validation set as `eval_set` to `fit()`; `evals_result()` returns the metric per round.

In [ ]:
X, y = diabetes.data, diabetes.target
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.25, random_state=42)

# One model without early stopping, to see the full train/validation curves
model_lc = xgb.XGBRegressor(objective="reg:squarederror", n_estimators=300, learning_rate=0.1,
                            max_depth=3, eval_metric="rmse", random_state=42)
model_lc.fit(X_train, y_train, eval_set=[(X_train, y_train), (X_val, y_val)], verbose=False)
curves = model_lc.evals_result()   # 'validation_0' = train set, 'validation_1' = validation set
val_rmse = curves['validation_1']['rmse']
best = int(np.argmin(val_rmse))
print(f"No early stopping: best val RMSE {val_rmse[best]:.1f} at round {best + 1}, "
      f"val RMSE after 300 rounds {val_rmse[-1]:.1f}\n")

# Several learning rates, each with early stopping
stopped = {}
for lr in [0.3, 0.1, 0.03]:
    m = xgb.XGBRegressor(objective="reg:squarederror", n_estimators=2000, learning_rate=lr,
                         max_depth=3, eval_metric="rmse", early_stopping_rounds=50,
                         random_state=42)
    m.fit(X_train, y_train, eval_set=[(X_val, y_val)], verbose=False)
    stopped[lr] = m
    print(f"learning_rate={lr:<5}  best round: {m.best_iteration + 1:3d}   best val RMSE: {m.best_score:.1f}")

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 4))

ax1.plot(curves['validation_0']['rmse'], lw=2, label='train')
ax1.plot(val_rmse, lw=2, label='validation')
ax1.axvline(best, color='gray', ls=':', label=f'best validation round ({best + 1})')
ax1.set_xlabel('boosting round'); ax1.set_ylabel('RMSE')
ax1.set_title(r'Train vs. validation RMSE ($\eta$=0.1)'); ax1.legend()

for lr, m in stopped.items():
    ax2.plot(m.evals_result()['validation_0']['rmse'], lw=2, label=rf'$\eta$={lr}')
    ax2.scatter(m.best_iteration, m.best_score, color='k', zorder=3)
ax2.set_ylim(50, 80)
ax2.set_xlabel('boosting round'); ax2.set_ylabel('validation RMSE')
ax2.set_title('Smaller learning rate = more rounds (dots: best round)'); ax2.legend()

plt.tight_layout()
plt.show()

The training error keeps falling while the validation error reaches a minimum after about 20 rounds and then rises: that is overfitting. Early stopping picks the round with the lowest validation error. A smaller learning rate needs more rounds to get there; on this small dataset all three reach a similar best RMSE.

After early stopping, `predict()` automatically uses the trees up to `best_iteration`. To train a final model on all the data (train + validation), use `n_estimators = best_iteration + 1`.

## 7. Feature importance

XGBoost can rank features by how often they are used in splits (`weight`), or by how much they reduce the loss on average (`gain`, usually more informative). Both are computed on training data and split the credit between correlated features; permutation importance on held-out data is more trustworthy (see notes 07, section 5).

In [ ]:
# The breast cancer classifier from section 3
xgb_clf.get_booster().feature_names = list(cancer.feature_names)   # readable labels in the plot

fig, ax = plt.subplots(figsize=(8, 5))
xgb.plot_importance(xgb_clf, ax=ax, importance_type='gain', max_num_features=10,
                    show_values=False, title='Breast cancer: top 10 features by gain')
plt.tight_layout()
plt.show()